In [1]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("RawDataInspection")
  .getOrCreate()

// קריאת הקובץ (עדכני את הנתיב לפי הצורך)
val df = spark.read
  .option("header", "true")
  .option("inferSchema", "true")
  .option("mode", "PERMISSIVE")  // גישה סלחנית לשורות שגויות
  .csv("hdfs://namenode:8020/data/datasetN3.txt")

// הצגת 5 שורות ראשונות
df.show(5)

// כמה שורות יש בסך הכול?
println(s"Total rows: ${df.count()}")

+-----------+---+-----------+------+-------+---------------------+--------+----------------------+--------------+--------------+
|row  number|age|work sector|salery|savings|Food expenses (month)|zip code|other expenses (month)|educaion years|economic class|
+-----------+---+-----------+------+-------+---------------------+--------+----------------------+--------------+--------------+
|          1| 37|          B|  6643|  35501|                 1185|   24602|                  1958|            18|             5|
|          2| 35|          E|  7580|  28599|                 1960|   21555|                  1732|            19|             6|
|          3| 29|          B|  7612|  21559|                 1155|   31315|                  1294|            18|             7|
|          4| 52|          E|  6958|  44584|                 1760|   24157|                  1157|            17|             6|
|          5| 38|          D|  2818|   6724|                  470|   18148|                  1935

[row  number: int, age: int ... 8 more fields]

In [2]:
val cleanedColumns = Seq(
  "age",
  "work_sector",
  "salary",         // תיקון מ-salery
  "savings",
  "food_expenses_month",    // תיקון ושיפור שם
  "zip_code",
  "other_expenses_month",
  "education_years",        // תיקון שגיאת כתיב
  "economic_class"
)
// הסרת "row  number" ושינוי שמות שאר העמודות
val df_renamed = df.drop("row  number")
  .toDF(cleanedColumns: _*)

df_renamed.printSchema()
df_renamed.show(5)

root
 |-- age: integer (nullable = true)
 |-- work_sector: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- savings: integer (nullable = true)
 |-- food_expenses_month: integer (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- other_expenses_month: integer (nullable = true)
 |-- education_years: integer (nullable = true)
 |-- economic_class: integer (nullable = true)

+---+-----------+------+-------+-------------------+--------+--------------------+---------------+--------------+
|age|work_sector|salary|savings|food_expenses_month|zip_code|other_expenses_month|education_years|economic_class|
+---+-----------+------+-------+-------------------+--------+--------------------+---------------+--------------+
| 37|          B|  6643|  35501|               1185|   24602|                1958|             18|             5|
| 35|          E|  7580|  28599|               1960|   21555|                1732|             19|             6|
| 29|          B|  7612| 

cleanedColumns = List(age, work_sector, salary, savings, food_expenses_month, zip_code, other_expenses_month, education_years, economic_class)
df_renamed = [age: int, work_sector: string ... 7 more fields]


[age: int, work_sector: string ... 7 more fields]

In [3]:
import org.apache.spark.sql.functions._

// 1. תנאי לזיהוי שורות ריקות
val allColumns = df_renamed.columns
val emptyRowCondition = allColumns.map { colName =>
  col(colName).isNull || trim(col(colName)) === ""
}.reduce(_ && _)

// 2. סינון אמיתי של הדאטה – לא רק בדיקה
val df_no_empty = df_renamed.filter(!emptyRowCondition)

// 3. שמירה בזיכרון (persist) כי נשתמש ב־df_no_empty הרבה בהמשך
df_no_empty.persist()
df_no_empty.count()  // חימום הקאש

// 4. בדיקה לוודא שנשמר כמו שצריך
df_no_empty.show(5)

+---+-----------+------+-------+-------------------+--------+--------------------+---------------+--------------+
|age|work_sector|salary|savings|food_expenses_month|zip_code|other_expenses_month|education_years|economic_class|
+---+-----------+------+-------+-------------------+--------+--------------------+---------------+--------------+
| 37|          B|  6643|  35501|               1185|   24602|                1958|             18|             5|
| 35|          E|  7580|  28599|               1960|   21555|                1732|             19|             6|
| 29|          B|  7612|  21559|               1155|   31315|                1294|             18|             7|
| 52|          E|  6958|  44584|               1760|   24157|                1157|             17|             6|
| 38|          D|  2818|   6724|                470|   18148|                1935|             10|             1|
+---+-----------+------+-------+-------------------+--------+--------------------+------

allColumns = Array(age, work_sector, salary, savings, food_expenses_month, zip_code, other_expenses_month, education_years, economic_class)
emptyRowCondition = ((((((((((age IS NULL) OR (trim(age) = )) AND ((work_sector IS NULL) OR (trim(work_sector) = ))) AND ((salary IS NULL) OR (trim(salary) = ))) AND ((savings IS NULL) OR (trim(savings) = ))) AND ((food_expenses_month IS NULL) OR (trim(food_expenses_month) = ))) AND ((zip_code IS NULL) OR (trim(zip_code) = ))) AND ((other_expenses_month IS NULL) OR (trim(other_expenses_month) = ))) AND ((education_years IS NULL) OR (trim(education_years) = ))) AND ((economic_class IS NULL) OR (trim(economic_class) = )))


df_no_empty: org.apache.spark.sql.Dataset[org.a...


((((((((((age IS NULL) OR (trim(age) = )) AND ((work_sector IS NULL) OR (trim(work_sector) = ))) AND ((salary IS NULL) OR (trim(salary) = ))) AND ((savings IS NULL) OR (trim(savings) = ))) AND ((food_expenses_month IS NULL) OR (trim(food_expenses_month) = ))) AND ((zip_code IS NULL) OR (trim(zip_code) = ))) AND ((other_expenses_month IS NULL) OR (trim(other_expenses_month) = ))) AND ((education_years IS NULL) OR (trim(education_years) = ))) AND ((economic_class IS NULL) OR (trim(economic_class) = )))

### טבלת חוסרים – Missing Values Analysis

בשלב זה סרקנו את כל העמודות והצגנו את כמות ואחוז הערכים החסרים (NULL או ריקים).

טבלה זו מאפשרת:
- להבין אילו עמודות דורשות טיפול (לדוגמה: השלמה, השערה, או הסרה)
- לזהות עמודות בעייתיות שעלולות להשפיע על המודל

שורות ריקות לגמרי הוסרו מראש, ולכן הניתוח מתבסס על שורות תקפות בלבד.

In [4]:
import org.apache.spark.sql.functions._

val totalRows = df_no_empty.count()

val missingStats = df_no_empty.columns.map { colName =>
  val missingCount = df_no_empty.filter(col(colName).isNull || trim(col(colName)) === "").count()
  val missingPct = (missingCount.toDouble / totalRows) * 100
  (colName, missingCount, f"$missingPct%.2f%%")
}.toSeq.toDF("column", "missing_count", "missing_percent")

// הצגה ממוינת
missingStats.orderBy(desc("missing_count")).show(truncate = false)

totalRows = 5986
missingStats = [column: string, missing_count: bigint ... 1 more field]


+--------------------+-------------+---------------+
|column              |missing_count|missing_percent|
+--------------------+-------------+---------------+
|economic_class      |46           |0.77%          |
|age                 |18           |0.30%          |
|food_expenses_month |18           |0.30%          |
|zip_code            |18           |0.30%          |
|other_expenses_month|16           |0.27%          |
|education_years     |16           |0.27%          |
|work_sector         |11           |0.18%          |
|salary              |0            |0.00%          |
|savings             |0            |0.00%          |
+--------------------+-------------+---------------+



[column: string, missing_count: bigint ... 1 more field]

In [5]:
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types.NumericType

// אם את רוצה לבדוק רק על שורות שיש להן economic class:
val dfToCheck = df_no_empty.filter(col("economic class").isNotNull)
// ואם את רוצה על כל הטבלה בלי סינון:
// val dfToCheck = df_no_empty

// מוצאים את כל העמודות הנומריות (גם אם השמות עם רווחים)
val numericCols: Seq[String] =
  dfToCheck.schema.fields.collect {
    case f if f.dataType.isInstanceOf[NumericType] => f.name
  }.toSeq

println("Numeric columns: " + numericCols.mkString(", "))

// בונים רשימת אגרגציות: min(col), max(col) לכל עמודה נומרית
val aggExprs = numericCols.flatMap { c =>
  Seq(
    min(col(c)).alias(s"${c}_min"),
    max(col(c)).alias(s"${c}_max")
  )
}

// מריצים אגרגציה אחת שמחזירה שורה עם כל המינימום/מקסימום
dfToCheck.agg(aggExprs.head, aggExprs.tail: _*).show(false)

Numeric columns: age, salary, savings, food_expenses_month, zip_code, other_expenses_month, education_years, economic_class
+-------+-------+----------+----------+-----------+-----------+-----------------------+-----------------------+------------+------------+------------------------+------------------------+-------------------+-------------------+------------------+------------------+
|age_min|age_max|salary_min|salary_max|savings_min|savings_max|food_expenses_month_min|food_expenses_month_max|zip_code_min|zip_code_max|other_expenses_month_min|other_expenses_month_max|education_years_min|education_years_max|economic_class_min|economic_class_max|
+-------+-------+----------+----------+-----------+-----------+-----------------------+-----------------------+------------+------------+------------------------+------------------------+-------------------+-------------------+------------------+------------------+
|2      |76     |263       |10603     |26         |136112     |302            

dfToCheck = [age: int, work_sector: string ... 7 more fields]
numericCols = WrappedArray(age, salary, savings, food_expenses_month, zip_code, other_expenses_month, education_years, economic_class)
aggExprs = ArrayBuffer(min(age) AS age_min, max(age) AS age_max, min(salary) AS salary_min, max(salary) AS salary_max, min(savings) AS savings_min, max(savings) AS savings_max, min(food_expenses_month) AS food_expenses_month_min, max(food_expenses_month) AS food_expenses_month_max, min(zip_code) AS zip_code_min, max(zip_code) AS zip_code_max, min(other_expenses_month) AS other_expenses_month_min, max(other_e...


ArrayBuffer(min(age) AS age_min, max(age) AS age_max, min(salary) AS salary_min, max(salary) AS salary_max, min(savings) AS savings_min, max(savings) AS savings_max, min(food_expenses_month) AS food_expenses_month_min, max(food_expenses_month) AS food_expenses_month_max, min(zip_code) AS zip_code_min, max(zip_code) AS zip_code_max, min(other_expenses_month) AS other_expenses_month_min, max(other_e...

In [6]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

// הגדרת חלון דירוג לפי שכר
val windowSpec = Window.orderBy("salary")

// הוספת דירוג יחסי (percent_rank)
val df_with_rank = df_no_empty.withColumn("salary_rank", percent_rank().over(windowSpec))

// המרת הדירוג לעשירון: 0–9
val df_with_decile = df_with_rank.withColumn("salary_decile", (col("salary_rank") * 10).cast("int"))

// הסרת עמודת עזר: salary_rank
val df_final = df_with_decile.drop("salary_rank")

// שמירה בזיכרון, כי נשתמש בזה בהמשך (EDA, פיצ’רים, מודל)
df_final.persist()
df_final.count()  // פעולה שמריצה את השרשרת ומחממת את ה־cache

// הצגה לבדיקה
df_final.select("age", "salary", "salary_decile", "economic_class").show(10, truncate = false)

+---+------+-------------+--------------+
|age|salary|salary_decile|economic_class|
+---+------+-------------+--------------+
|55 |263   |0            |0             |
|38 |635   |0            |0             |
|74 |707   |0            |0             |
|52 |732   |0            |0             |
|68 |747   |0            |0             |
|46 |779   |0            |0             |
|28 |789   |0            |0             |
|35 |850   |0            |0             |
|41 |868   |0            |0             |
|28 |887   |0            |0             |
+---+------+-------------+--------------+
only showing top 10 rows



windowSpec = org.apache.spark.sql.expressions.WindowSpec@64a6daac
df_with_rank = [age: int, work_sector: string ... 8 more fields]
df_with_decile = [age: int, work_sector: string ... 9 more fields]
df_final = [age: int, work_sector: string ... 8 more fields]


[age: int, work_sector: string ... 8 more fields]

In [7]:
// ממוצע הוצאות אוכל והוצאות אחרות לפי עשירון שכר
val expenses_by_decile = df_final
  .groupBy("salary_decile")
  .agg(
    round(avg("food_expenses_month")).alias("avg_food_expenses"),
    round(avg("other_expenses_month")).alias("avg_other_expenses"),
    count("*").alias("people_in_decile")
  )
  .orderBy("salary_decile")

expenses_by_decile.show(10, truncate = false)

+-------------+-----------------+------------------+----------------+
|salary_decile|avg_food_expenses|avg_other_expenses|people_in_decile|
+-------------+-----------------+------------------+----------------+
|0            |506.0            |1667.0            |599             |
|1            |620.0            |1703.0            |599             |
|2            |742.0            |1691.0            |598             |
|3            |782.0            |1737.0            |598             |
|4            |971.0            |1706.0            |599             |
|5            |1034.0           |1715.0            |598             |
|6            |1216.0           |1716.0            |599             |
|7            |1418.0           |1698.0            |598             |
|8            |1520.0           |1700.0            |599             |
|9            |1569.0           |1713.0            |598             |
+-------------+-----------------+------------------+----------------+
only showing top 10 

expenses_by_decile = [salary_decile: int, avg_food_expenses: double ... 2 more fields]


[salary_decile: int, avg_food_expenses: double ... 2 more fields]

In [8]:
val education_by_decile = df_final
  .groupBy("salary_decile")
  .agg(
    round(avg("education_years")).alias("avg_education_years"),
    count("*").alias("people_in_decile")
  )
  .orderBy("salary_decile")

education_by_decile.show(10, truncate = false)

+-------------+-------------------+----------------+
|salary_decile|avg_education_years|people_in_decile|
+-------------+-------------------+----------------+
|0            |13.0               |599             |
|1            |13.0               |599             |
|2            |14.0               |598             |
|3            |14.0               |598             |
|4            |16.0               |599             |
|5            |17.0               |598             |
|6            |17.0               |599             |
|7            |18.0               |598             |
|8            |18.0               |599             |
|9            |18.0               |598             |
+-------------+-------------------+----------------+
only showing top 10 rows



education_by_decile = [salary_decile: int, avg_education_years: double ... 1 more field]


[salary_decile: int, avg_education_years: double ... 1 more field]

In [9]:
val education_by_sector = df_final
  .where(col("work_sector").isNotNull) // סינון חסרים
  .groupBy("work_sector")
  .agg(
    round(avg("education_years")).alias("avg_education_years"),
    count("*").alias("people_in_sector")
  )
  .orderBy("work_sector")

education_by_sector.show(truncate = false)

+-----------+-------------------+----------------+
|work_sector|avg_education_years|people_in_sector|
+-----------+-------------------+----------------+
|A          |16.0               |1178            |
|B          |17.0               |1193            |
|C          |14.0               |1304            |
|D          |13.0               |1139            |
|E          |18.0               |1161            |
+-----------+-------------------+----------------+



education_by_sector = [work_sector: string, avg_education_years: double ... 1 more field]


[work_sector: string, avg_education_years: double ... 1 more field]

In [10]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

// =======================
// שלב 0: הוספת מזהה שורה למעקב
// =======================
val df_indexed = df_final.withColumn("row_id", monotonically_increasing_id())

// =======================
// שלב 1: ממוצע שנות לימוד לפי עשירון וסקטור
// =======================
val avgEduByGroup = df_indexed
  .filter(col("education_years").isNotNull)
  .groupBy("work_sector", "salary_decile")
  .agg(round(avg("education_years"), 2).alias("avg_edu_sector_decile"))

val df_with_avg_edu = df_indexed.join(
  avgEduByGroup,
  Seq("work_sector", "salary_decile"),
  "left"
)

// =======================
// שלב 2: מילוי עמודת education_years החסרה לפי ממוצע
// =======================
val df_filled_edu = df_with_avg_edu
  .withColumn("education_years_filled_raw",
    when(col("education_years").isNull, col("avg_edu_sector_decile"))
    .otherwise(col("education_years"))
  )
  .withColumn("education_years_filled", round(col("education_years_filled_raw")).cast("int"))
  .drop("avg_edu_sector_decile", "education_years_filled_raw")

// =======================
// שלב 3: מציאת הסקטור הנפוץ ביותר לפי עשירון ושנות לימוד ממולאות
// =======================
val sectorCounts = df_filled_edu
  .groupBy("salary_decile", "education_years_filled", "work_sector")
  .agg(count("*").alias("count"))

val sectorWindow = Window.partitionBy("salary_decile", "education_years_filled").orderBy(desc("count"))

val sectorMode = sectorCounts
  .withColumn("rank", row_number().over(sectorWindow))
  .filter(col("rank") === 1)
  .select("salary_decile", "education_years_filled", "work_sector")
  .withColumnRenamed("work_sector", "most_common_sector")

// =======================
// שלב 4: מילוי work_sector לפי הערך הנפוץ
// =======================
val df_with_mode_sector = df_filled_edu.join(
  sectorMode,
  Seq("salary_decile", "education_years_filled"),
  "left"
)

val df_filled_all = df_with_mode_sector
  .withColumn("work_sector_filled",
    when(col("work_sector").isNull, col("most_common_sector"))
    .otherwise(col("work_sector"))
  )

// =======================
// שלב 5: תצוגת דוגמאות למילוי
// =======================
println("🔧 דוגמה לשורות שמולאו בעמודת work_sector:")
df_filled_all
  .filter(col("work_sector").isNull && col("work_sector_filled").isNotNull)
  .select("salary_decile", "education_years_filled", "work_sector_filled")
  .distinct()
  .show(5, truncate = false)

println("🔧 דוגמה לשורות שמולאו בעמודת education_years:")
df_filled_all
  .filter(col("education_years").isNull && col("education_years_filled").isNotNull)
  .select("salary_decile", "work_sector", "education_years_filled")
  .distinct()
  .show(5, truncate = false)

// =======================
// שלב 6: סטטיסטיקת מילוי
// =======================
val sectorFillCount = df_filled_all.filter(col("work_sector").isNull && col("work_sector_filled").isNotNull).count()
val eduFillCount = df_filled_all.filter(col("education_years").isNull && col("education_years_filled").isNotNull).count()

println(s"✅ מולאו $sectorFillCount ערכים חסרים בעמודת work_sector")
println(s"✅ מולאו $eduFillCount ערכים חסרים בעמודת education_years")

🔧 דוגמה לשורות שמולאו בעמודת work_sector:
+-------------+----------------------+------------------+
|salary_decile|education_years_filled|work_sector_filled|
+-------------+----------------------+------------------+
|0            |12                    |D                 |
|0            |13                    |D                 |
|1            |11                    |D                 |
|1            |13                    |D                 |
|1            |15                    |C                 |
+-------------+----------------------+------------------+
only showing top 5 rows

🔧 דוגמה לשורות שמולאו בעמודת education_years:
+-------------+-----------+----------------------+
|salary_decile|work_sector|education_years_filled|
+-------------+-----------+----------------------+
|0            |D          |13                    |
|1            |D          |12                    |
|2            |C          |14                    |
|3            |D          |12                    |
|4      

df_indexed = [age: int, work_sector: string ... 9 more fields]
avgEduByGroup = [work_sector: string, salary_decile: int ... 1 more field]
df_with_avg_edu = [work_sector: string, salary_decile: int ... 10 more fields]
df_filled_edu = [work_sector: string, salary_decile: int ... 10 more fields]
sectorCounts = [salary_decile: int, education_years_filled: int ... 2 more fields]
sectorWindow = org.apache.spark.sql.expressions.WindowSpec@49f2e25c
sectorMode = [salary_decile:...


[salary_decile:...